<a href="https://colab.research.google.com/github/Yisihaq01/breast-histology-calibration/blob/main/Notebooks/03_breakhis_formal_development.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Formal Development-Only Model Selection

## CIS 631 — Breast Histopathology Calibration Research

This notebook implements the formal model-development stage of our breast histopathology calibration study.

The objective is to compare fine-tuned ResNet-18 and EfficientNet-B0 using the previously established patient-grouped three-fold Development partition.

### Experimental restrictions

- Only the **49 BreakHis Development patients** can be used for CNN model development and model selection.
- Only BreakHis **40× images** are used for primary model development.
- Existing patient assignments, Development folds, preprocessing parameters, and patch coordinates must remain unchanged.
- Calibration patients are reserved for fitting post-hoc calibrators after final CNN training.
- Test patients and external BACH/BRACS datasets are excluded from model development.
- Model selection will be based on Development-fold validation results.

The preprocessing parameters remain fixed:

| Parameter | Value |
|---|---|
| Source physical resolution | 0.49 µm/pixel |
| Target physical resolution | 0.50 µm/pixel |
| Patch size | 224 × 224 |
| Grid | 4 × 4 |
| Maximum patches per image | 16 |
| Optical density threshold | 0.08 |
| Saturation threshold | 0.05 |
| Minimum tissue fraction | 0.20 |

### Stage A: Data and fold verification

Before training, we reconstruct the complete Development patch manifest from previously saved patch manifests and verify that all three folds preserve patient-level separation.

No CNN training or held-out performance evaluation is performed during Stage A.


In [5]:

# ============================================================
# Stage A — Environment and repository setup
# ============================================================

from pathlib import Path
import subprocess
import random

import numpy as np
import pandas as pd

from IPython.display import display

SEED = 631

random.seed(SEED)
np.random.seed(SEED)

REPO_ROOT = Path(
    "/content/breast-histology-calibration"
)

REPO_URL = (
    "https://github.com/Yisihaq01/"
    "breast-histology-calibration.git"
)

if not REPO_ROOT.exists():
    subprocess.run(
        [
            "git",
            "clone",
            REPO_URL,
            str(REPO_ROOT)
        ],
        check=True
    )
else:
    print(
        "Repository folder already exists."
    )

METADATA_DIR = REPO_ROOT / "Metadata"

RESULTS_DIR = (
    REPO_ROOT /
    "results" /
    "formal_development"
)

RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("Repository:", REPO_ROOT)
print("Seed:", SEED)
print("Metadata directory:", METADATA_DIR)
print("Results directory:", RESULTS_DIR)


Repository folder already exists.
Repository: /content/breast-histology-calibration
Seed: 631
Metadata directory: /content/breast-histology-calibration/Metadata
Results directory: /content/breast-histology-calibration/results/formal_development


In [7]:

# ============================================================
# Stage A — Inspect frozen metadata structure
# ============================================================

import pandas as pd

for name, filepath in required_files.items():

    df = pd.read_csv(filepath)

    print("\n" + "=" * 55)
    print(name.upper())
    print("=" * 55)

    print("Rows:", len(df))
    print("Columns:", df.columns.tolist())

    if name == "development_folds":
        print("\nFirst 5 fold assignments:")
        display(df.head(5))

    elif name == "patient_split":
        print("\nPatient split counts:")
        print(df["split"].value_counts())

    elif name in [
        "training_patch_manifest",
        "validation_patch_manifest"
    ]:
        print("\nFirst 2 patch records:")
        display(df.head(2))



FINAL_40X_METADATA
Rows: 1961
Columns: ['filename', 'relative_path', 'patient_id', 'diagnosis_code', 'diagnosis', 'original_subtype_code', 'original_subtype', 'subtype_code', 'subtype', 'magnification', 'image_number', 'split']

PATIENT_SPLIT
Rows: 81
Columns: ['patient_id', 'diagnosis_code', 'diagnosis', 'subtype_code', 'subtype', 'split']

Patient split counts:
split
development    49
calibration    16
test           16
Name: count, dtype: int64

DEVELOPMENT_FOLDS
Rows: 49
Columns: ['patient_id', 'diagnosis_code', 'diagnosis', 'subtype_code', 'subtype', 'split', 'development_fold']

First 5 fold assignments:


,patient_id,diagnosis_code,diagnosis,subtype_code,subtype,split,development_fold
0,14-29960CD,B,Benign,A,Adenosis,development,1
1,14-14134,B,Benign,F,Fibroadenoma,development,1
2,14-21998CD,B,Benign,F,Fibroadenoma,development,1
3,14-13200,B,Benign,TA,Tubular adenoma,development,1
4,14-3411F,B,Benign,TA,Tubular adenoma,development,1



TRAINING_PATCH_MANIFEST
Rows: 12034
Columns: ['patient_id', 'filename', 'relative_path', 'diagnosis', 'label', 'patch_index', 'x', 'y', 'tissue_fraction', 'used_fallback']

First 2 patch records:


,patient_id,filename,relative_path,diagnosis,label,patch_index,x,y,tissue_fraction,used_fallback
0,14-10147,SOB_M_MC-14-10147-40-009.png,BreaKHis_v1/BreaKHis_v1/histology_slides/breas...,Malignant,1,0,0,0,0.578823,False
1,14-10147,SOB_M_MC-14-10147-40-009.png,BreaKHis_v1/BreaKHis_v1/histology_slides/breas...,Malignant,1,1,154,0,0.518634,False



VALIDATION_PATCH_MANIFEST
Rows: 5861
Columns: ['patient_id', 'filename', 'relative_path', 'diagnosis', 'label', 'patch_index', 'x', 'y', 'tissue_fraction', 'used_fallback']

First 2 patch records:


,patient_id,filename,relative_path,diagnosis,label,patch_index,x,y,tissue_fraction,used_fallback
0,14-13200,SOB_B_TA-14-13200-40-012.png,BreaKHis_v1/BreaKHis_v1/histology_slides/breas...,Benign,0,0,0,0,0.928133,False
1,14-13200,SOB_B_TA-14-13200-40-012.png,BreaKHis_v1/BreaKHis_v1/histology_slides/breas...,Benign,0,1,154,0,0.830716,False


In [4]:
# ============================================================
# Check which GitHub branches contain the frozen metadata
# ============================================================

import subprocess
from pathlib import Path

REPO_ROOT = Path(
    "/content/breast-histology-calibration"
)

required_names = [
    "breakhis_40x_final_metadata.csv",
    "breakhis_patient_split.csv",
    "breakhis_development_folds.csv",
    "baseline_train_patch_manifest.csv",
    "baseline_val_patch_manifest.csv",
]

branches = [
    "origin/main",
    "origin/Calibration_analysis"
]

for branch in branches:

    result = subprocess.run(
        [
            "git",
            "-C", str(REPO_ROOT),
            "ls-tree",
            "-r",
            "--name-only",
            branch
        ],
        capture_output=True,
        text=True,
        check=True
    )

    tracked_files = result.stdout.splitlines()

    print(f"\n===== {branch} =====")

    for filename in required_names:

        matches = [
            path for path in tracked_files
            if Path(path).name == filename
        ]

        if matches:
            print("FOUND:", matches[0])
        else:
            print("MISSING:", filename)

# Check if Git is ignoring the metadata directory
print("\n===== Git ignore check =====")

result = subprocess.run(
    [
        "git",
        "-C", str(REPO_ROOT),
        "check-ignore",
        "-v",
        "--no-index",
        "metadata/breakhis_patient_split.csv"
    ],
    capture_output=True,
    text=True
)

print(
    result.stdout.strip()
    or "No matching ignore rule found."
)


===== origin/main =====
FOUND: Metadata/breakhis_40x_final_metadata.csv
FOUND: Metadata/breakhis_patient_split.csv
FOUND: Metadata/breakhis_development_folds.csv
FOUND: Metadata/baseline_train_patch_manifest.csv
FOUND: Metadata/baseline_val_patch_manifest.csv

===== origin/Calibration_analysis =====
FOUND: Metadata/breakhis_40x_final_metadata.csv
FOUND: Metadata/breakhis_patient_split.csv
FOUND: Metadata/breakhis_development_folds.csv
FOUND: Metadata/baseline_train_patch_manifest.csv
FOUND: Metadata/baseline_val_patch_manifest.csv

===== Git ignore check =====
No matching ignore rule found.


In [9]:

# ============================================================
# Load and verify frozen Development metadata
# ============================================================

import pandas as pd

image_metadata = pd.read_csv(
    required_files["final_40x_metadata"],
    dtype={"patient_id": str}
)

patient_split = pd.read_csv(
    required_files["patient_split"],
    dtype={"patient_id": str}
)

development_folds = pd.read_csv(
    required_files["development_folds"],
    dtype={"patient_id": str}
)

train_patches = pd.read_csv(
    required_files["training_patch_manifest"],
    dtype={"patient_id": str}
)

val_patches = pd.read_csv(
    required_files["validation_patch_manifest"],
    dtype={"patient_id": str}
)

# Standardize labels without changing stored files
for df in [image_metadata, patient_split, development_folds]:
    df["split"] = df["split"].str.strip().str.lower()

development_folds["development_fold"] = (
    development_folds["development_fold"].astype(int)
)

# Verify original data dimensions
assert len(image_metadata) == 1961
assert len(patient_split) == 81
assert len(development_folds) == 49

assert len(train_patches) == 12034
assert len(val_patches) == 5861

assert image_metadata["patient_id"].nunique() == 81
assert patient_split["patient_id"].nunique() == 81
assert development_folds["patient_id"].nunique() == 49

assert not image_metadata["filename"].duplicated().any()
assert not patient_split["patient_id"].duplicated().any()
assert not development_folds["patient_id"].duplicated().any()

assert image_metadata["magnification"].astype(str).eq("40").all()

# Frozen split membership
DEV_IDS = set(
    patient_split.loc[
        patient_split["split"] == "development",
        "patient_id"
    ]
)

CAL_IDS = set(
    patient_split.loc[
        patient_split["split"] == "calibration",
        "patient_id"
    ]
)

TEST_IDS = set(
    patient_split.loc[
        patient_split["split"] == "test",
        "patient_id"
    ]
)

assert len(DEV_IDS) == 49
assert len(CAL_IDS) == 16
assert len(TEST_IDS) == 16

assert DEV_IDS.isdisjoint(CAL_IDS)
assert DEV_IDS.isdisjoint(TEST_IDS)
assert CAL_IDS.isdisjoint(TEST_IDS)

assert (
    DEV_IDS | CAL_IDS | TEST_IDS
) == set(patient_split["patient_id"])

# Check image split agrees with authoritative patient split
split_lookup = patient_split.set_index("patient_id")["split"]

assert (
    image_metadata["split"]
    == image_metadata["patient_id"].map(split_lookup)
).all()

# Development subset
dev_images = image_metadata.loc[
    image_metadata["patient_id"].isin(DEV_IDS)
].copy()

assert len(dev_images) == 1133
assert dev_images["patient_id"].nunique() == 49

# Verify original Development folds
assert set(development_folds["patient_id"]) == DEV_IDS
assert development_folds["split"].eq("development").all()

fold_counts = (
    development_folds["development_fold"]
    .value_counts()
    .sort_index()
)

assert fold_counts.to_dict() == {
    1: 17,
    2: 16,
    3: 16
}

print("Development patients:", len(DEV_IDS))
print("Calibration patients:", len(CAL_IDS))
print("Test patients:", len(TEST_IDS))

print("\nDevelopment images:", len(dev_images))

print("\nDevelopment fold counts:")
display(fold_counts)

print("\nPASS: Frozen Development metadata verified.")


Development patients: 49
Calibration patients: 16
Test patients: 16

Development images: 1133

Development fold counts:


,count
development_fold,
1,17
2,16
3,16



PASS: Frozen Development metadata verified.


In [10]:

# ============================================================
# Reconstruct Development patch manifest
# ============================================================

all_patches = pd.concat(
    [train_patches, val_patches],
    ignore_index=True
)

# Check total number of patches and membership
assert len(all_patches) == 17895

assert all_patches["patient_id"].nunique() == 49
assert all_patches["filename"].nunique() == 1133

assert set(all_patches["patient_id"]) == DEV_IDS
assert set(all_patches["filename"]) == set(dev_images["filename"])

# Check for repeated patch identities
assert not all_patches.duplicated(
    ["filename", "patch_index"]
).any()

# Build authoritative image lookup
image_lookup = dev_images.set_index("filename")

for column in [
    "patient_id",
    "diagnosis",
    "relative_path"
]:

    expected = all_patches["filename"].map(
        image_lookup[column]
    )

    assert expected.notna().all(), (
        f"Missing expected values for {column}"
    )

    assert (
        all_patches[column].astype(str).to_numpy()
        == expected.astype(str).to_numpy()
    ).all(), (
        f"Patch/image mismatch in {column}"
    )

# Verify binary labels
expected_labels = (
    all_patches["filename"]
    .map(image_lookup["diagnosis"])
    .eq("Malignant")
    .astype(int)
)

assert (
    all_patches["label"].astype(int).to_numpy()
    == expected_labels.to_numpy()
).all()

# Verify all patches use valid indices
assert all_patches["patch_index"].between(0, 15).all()
assert all_patches["tissue_fraction"].between(0, 1).all()

patches_per_image = (
    all_patches.groupby("filename").size()
)

assert patches_per_image.between(1, 16).all()

# Confirm that the old preliminary split used Fold 1
fold_lookup = development_folds.set_index(
    "patient_id"
)["development_fold"]

original_val_folds = set(
    val_patches["patient_id"].map(fold_lookup)
)

original_train_folds = set(
    train_patches["patient_id"].map(fold_lookup)
)

assert original_val_folds == {1}
assert original_train_folds == {2, 3}

assert set(train_patches["patient_id"]).isdisjoint(
    set(val_patches["patient_id"])
)

# Attach original frozen Development fold
all_patches["development_fold"] = (
    all_patches["patient_id"].map(fold_lookup)
)

assert all_patches["development_fold"].notna().all()

all_patches["development_fold"] = (
    all_patches["development_fold"].astype(int)
)

all_patches = all_patches.sort_values(
    [
        "development_fold",
        "patient_id",
        "filename",
        "patch_index"
    ]
).reset_index(drop=True)

print("Development patients:", all_patches["patient_id"].nunique())
print("Development images:", all_patches["filename"].nunique())
print("Development patches:", len(all_patches))

print("\nPatches per image:")
print(patches_per_image.describe())

print("\nPASS: Frozen Development patch manifest reconstructed.")


Development patients: 49
Development images: 1133
Development patches: 17895

Patches per image:
count    1133.000000
mean       15.794351
std         1.121223
min         1.000000
25%        16.000000
50%        16.000000
75%        16.000000
max        16.000000
dtype: float64

PASS: Frozen Development patch manifest reconstructed.


In [11]:

# ============================================================
# Prepare patient-grouped three-fold CV
# ============================================================

cv_splits = {}
cv_summary_rows = []

for validation_fold in [1, 2, 3]:

    train_df = all_patches.loc[
        all_patches["development_fold"] != validation_fold
    ].copy()

    val_df = all_patches.loc[
        all_patches["development_fold"] == validation_fold
    ].copy()

    train_patients = set(train_df["patient_id"])
    val_patients = set(val_df["patient_id"])

    # No shared patients
    assert train_patients.isdisjoint(val_patients)

    # Together they cover all Development patients
    assert train_patients | val_patients == DEV_IDS

    # No held-out patients
    assert (
        train_patients | val_patients
    ).isdisjoint(CAL_IDS | TEST_IDS)

    # Save each fold in memory for later model training
    cv_splits[validation_fold] = {
        "train": train_df,
        "validation": val_df
    }

    cv_summary_rows.append({
        "validation_fold": validation_fold,
        "training_patients": len(train_patients),
        "validation_patients": len(val_patients),
        "training_images": train_df["filename"].nunique(),
        "validation_images": val_df["filename"].nunique(),
        "training_patches": len(train_df),
        "validation_patches": len(val_df)
    })

cv_summary = pd.DataFrame(cv_summary_rows)

display(cv_summary)

assert cv_summary["training_patients"].tolist() == [
    32, 33, 33
]

assert cv_summary["validation_patients"].tolist() == [
    17, 16, 16
]

assert cv_summary["validation_images"].sum() == 1133

assert cv_summary["validation_patches"].sum() == 17895

print(
    "\nPASS: All three patient-grouped Development "
    "folds are ready."
)


,validation_fold,training_patients,validation_patients,training_images,validation_images,training_patches,validation_patches
0,1,32,17,761,372,12034,5861
1,2,33,16,702,431,11085,6810
2,3,33,16,803,330,12671,5224



PASS: All three patient-grouped Development folds are ready.


In [12]:

# ============================================================
# Save verified Development outputs
# ============================================================

from pathlib import Path

METADATA_DIR = REPO_ROOT / "Metadata"

RESULTS_DIR = (
    REPO_ROOT / "results" / "formal_development"
)

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

manifest_path = (
    METADATA_DIR /
    "breakhis_development_all_patch_manifest.csv"
)

summary_path = (
    RESULTS_DIR /
    "development_cv_preflight_summary.csv"
)

all_patches.to_csv(manifest_path, index=False)
cv_summary.to_csv(summary_path, index=False)

# Reopen files to verify they were written correctly
saved_manifest = pd.read_csv(
    manifest_path,
    dtype={"patient_id": str}
)

saved_summary = pd.read_csv(summary_path)

assert len(saved_manifest) == 17895
assert saved_manifest["patient_id"].nunique() == 49

assert len(saved_summary) == 3

print("Saved files:")
print(manifest_path)
print(summary_path)

print("\nPASS: Stage A completed successfully.")


Saved files:
/content/breast-histology-calibration/Metadata/breakhis_development_all_patch_manifest.csv
/content/breast-histology-calibration/results/formal_development/development_cv_preflight_summary.csv

PASS: Stage A completed successfully.
